# Imports

In [0]:
from pyspark.sql.functions import regexp_replace, trim, col
from pyspark.sql.types import StringType

# Read from bronze table

In [0]:
df = spark.table("bike_data.bronze.px_cat_g1v2_raw")

In [0]:
df.display()

# Transformations

## Trimming

In [0]:
for item in df.schema.fields:
    if isinstance(item.dataType, StringType):
        df = df.withColumn(item.name, trim(col(item.name)))

## Field Name Normalization

In [0]:
# snake_case → kebab-case
df = (
    df
    .withColumn("ID", regexp_replace(col("ID"), "_", "-"))
)

## Column Renaming

In [0]:
renamed_col ={
    "ID": "category_key",
    "CAT": "category",
    "SUBCAT": "subcategory",
    "MAINTENANCE": "maintenance"
}

In [0]:
for old_name, new_name in renamed_col.items():
    df = df.withColumnRenamed(old_name, new_name)

# Write to Silver table

In [0]:
(
    df.write
    .mode("overwrite")
    .saveAsTable("bike_data.silver.erp_prduct_category")
)